# Chapitre 7 - Cours 3 : validation croisée et bootstrap

Les critères analytiques du notebook précédent (Cp, AIC, BIC) supposent un modèle dont on
connaît le nombre (effectif) de paramètres. La **validation croisée** et le **bootstrap**
estiment directement l'erreur de prédiction en **réutilisant** l'échantillon, pour
n'importe quelle méthode et n'importe quelle perte (sections 7.10 à 7.12 du livre) :

1. validation croisée à $K$ blocs, choix de $K$ et courbe d'apprentissage (fig. 7.8) ;
2. courbe de validation croisée et règle « un écart-type » (fig. 7.9) ;
3. raccourci leave-one-out et validation croisée généralisée (GCV) ;
4. la **bonne et la mauvaise** manière de faire une validation croisée (fig. 7.10, 7.11) ;
5. le bootstrap : estimateurs naïf, leave-one-out, **.632** et .632+ ;
6. comparaison des méthodes de sélection (fig. 7.7 et 7.13) et erreur conditionnelle
   contre erreur espérée (section 7.12).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch07_selection as sel
set_style()

## 1. Validation croisée à $K$ blocs

On découpe aléatoirement les $N$ observations en $K$ blocs de tailles à peu près égales.
Pour le bloc $k$, on ajuste le modèle sur les $K-1$ autres blocs et l'on calcule l'erreur
de prédiction sur le bloc $k$, laissé de côté. Si $\kappa(i) \in \{1, \dots, K\}$ désigne
le bloc de l'observation $i$ et $\hat f^{-k}$ le modèle ajusté sans le bloc $k$,

$$\mathrm{CV}(\hat f) = \frac1N\sum_{i=1}^N L\big(y_i, \hat f^{-\kappa(i)}(x_i)\big). \tag{7.48}$$

Pour une famille de modèles $f(x, \alpha)$, on calcule $\mathrm{CV}(\hat f, \alpha)$ pour
chaque $\alpha$ (éq. (7.49)) et l'on retient $\hat\alpha$ qui la minimise ; le modèle final
$f(x, \hat\alpha)$ est ensuite réajusté sur **toutes** les données.

Le cas $K = N$ s'appelle **leave-one-out** (on laisse une observation de côté à chaque
fois). Notre implémentation `sel.cross_val_error` prend une fonction
`fit_predict(Xtr, ytr, Xte)` qui contient **toute** la procédure d'ajustement ; elle
renvoie l'estimation CV et l'erreur-type des $K$ erreurs de bloc.

### 1.1 Que vaut $K$ ? La courbe d'apprentissage (figure 7.8)

* Avec $K = N$, chaque modèle est ajusté sur $N-1$ observations : l'estimation est
  presque **sans biais** pour l'erreur d'un modèle ajusté sur $N$ observations, mais elle
  peut être très **variable** (les $N$ échantillons d'apprentissage se ressemblent
  énormément) et coûteuse.
* Avec $K = 5$ ou $10$, les modèles sont ajustés sur $80$ ou $90\,\%$ des données : la
  variance est plus faible mais l'estimation est **biaisée vers le haut** si la
  performance dépend encore beaucoup de la taille de l'échantillon.

Une **courbe d'apprentissage** représente l'erreur espérée en fonction de la taille $N$ de
l'échantillon d'apprentissage. Pour la régression linéaire par moindres carrés avec $p$
prédicteurs gaussiens centrés réduits indépendants et un bruit de variance $\sigma^2$, un
calcul classique (espérance d'une loi de Wishart inverse) donne la formule exacte

$$\mathrm{Err}(N) = \sigma^2\Big(1 + \frac{p}{N - p - 1}\Big), \qquad N > p + 1.$$

Nous la vérifions par simulation puis nous l'utilisons pour visualiser le biais de la
validation croisée.

In [2]:
rng = np.random.RandomState(0)
p_lc, sig = 10, 1.0
N_grid = np.arange(13, 201)
err_th = sig ** 2 * (1 + p_lc / (N_grid - p_lc - 1))
N_sim = [15, 20, 30, 50, 100, 200]
err_sim = []
for n in N_sim:
    e = []
    for _ in range(2000):
        X = rng.randn(n, p_lc)
        b = np.linalg.lstsq(X, X @ np.ones(p_lc) + sig * rng.randn(n), rcond=None)[0]
        e.append(sig ** 2 + np.sum((b - 1) ** 2))       # Err_T exacte (cf. cours 1)
    err_sim.append(np.mean(e))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(N_grid, err_th, color=BLUE, label=r"$\mathrm{Err}(N) = \sigma^2(1 + p/(N-p-1))$")
ax.plot(N_sim, err_sim, "o", color=BLUE, label="simulation")
for n0, c, txt in [(50, ORANGE, (60, 1.9)), (200, COLORS[2], (95, 1.5))]:
    n5 = int(0.8 * n0)
    e0, e5 = [sig ** 2 * (1 + p_lc / (m - p_lc - 1)) for m in (n0, n5)]
    ax.plot([n5, n0], [e5, e0], "s", color=c, ms=7)
    ax.annotate("N=%d : CV 5 blocs ajuste sur %d\nbiais %.3f" % (n0, n5, e5 - e0),
                (n5, e5), xytext=txt, color=c, fontsize=9,
                arrowprops=dict(arrowstyle="->", color=c))
ax.set_ylim(0.9, 3)
ax.set_xlabel("Taille de l'échantillon d'apprentissage $N$")
ax.set_ylabel("Erreur de prédiction espérée Err")
ax.set_title("Courbe d'apprentissage (régression linéaire, $p = 10$)")
ax.legend()
plt.show()

Avec $N = 200$, la courbe est presque plate entre $160$ et $200$ : la validation croisée à
5 blocs est quasiment sans biais. Avec $N = 50$, elle évalue en fait un modèle ajusté sur
$40$ observations, nettement moins bon : elle **surestime** l'erreur. Le livre recommande
$K = 5$ ou $10$ comme bon compromis biais-variance.